# **IEP Least-Cost Result Extraction to clusters**
<font color="#FBB800">Agricultural Cold Chain Access Planning (AgCAP) Tool</font><br>

-------
# Description
The main purpose of this notebook is to extract any new results from the Least Cost Electrification model to the clusters. <br>
It takes as input the raw result file and the cluster file and merges the latest results as new columns:
1) Read the raw input data files from the local directory (typically, these are big files and cannot be hosted on GitHub)
2) Clean, transform, and extract the raw data into the settlement dataset
3) Perform data integrity checks and formatting
4) Export the processed data to be ingested by the **data extraction notebook**


-------
# License and Copyright Notice

This notebook is part of the **AgCAP** project.

**Copyright (C) 2025 Sustainable Energy for All**

This program is free software: you can redistribute it and/or modify it under the terms of the **GNU Affero General Public License version 3 (AGPLv3)** as published by the Free Software Foundation.

This work is distributed in the hope that it will be useful, but WITHOUT ANY WARRANTY; without even the implied warranty of MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE. See the GNU Affero General Public License for more details.

You should have received a copy of the GNU Affero General Public License along with this program. If not, see <https://www.gnu.org/licenses/agpl-3.0.html>.

**[Optional: Add a link to source code archive for AGPL compliance]**
*Download Source Code for this Version:* [Link to GitHub Tag Archive]

In [1]:
# Copyright (C) 2025 Sustainable Energy for All
#
#This program is free software: you can redistribute it and/or modify it 
# under the terms of the **GNU Affero General Public License version 3 (AGPLv3)** 
# as published by the Free Software Foundation.
#
# See <https://www.gnu.org/licenses/agpl-3.0.html>.

## Import packages and functions

In [2]:
import sys
from pathlib import Path
import numpy as np
from datetime import datetime
from rasterstats import zonal_stats

# Load the autoreload extension
%load_ext autoreload
%autoreload 2

current_dir = Path.cwd()
project_root = current_dir.parent

# Insert the project root path into the system path, allowing the notebook to find the 'scripts' folder as a package.
sys.path.insert(0, str(project_root))

# Import functions
from scripts.functions import *
from scripts.app import *

## Setting the target coordinate system (Mandatory)

When calculating distances it is important to choose a coordinate system that represents distances correctly in your area of interst. The coordinate system that is given below is the World Mercator, these coordinate system works well for Sub Saharan Africa but the distortions get larger as you move away from the equator.

In order to select your own coordinate system go to [epsg.io](http://epsg.io/) and type in your area of interest, this will give you a list of coordinate systems to choose from. Once you have selected your coordinate system replace the numbers below with the numbers from your coordinate system **(keep the "EPSG" part)**.

**NOTE** When selecting your coordinate system make sure that you select a system with the unit of meters, this is indicated for all systems on [epsg.io](http://epsg.io/)

In [3]:
## Coordinate and projection systems
crs_WGS84 = pyproj.CRS("EPSG:4326")    # Originan WGS84 coordinate system
crs_proj = pyproj.CRS("EPSG:32736")    # Projection system for the selected country -- see http://epsg.io/ for more info 

## Import Cluster layer and IEP result file

In [4]:
def get_user_input_paths():
    csv_path = Path(input("Enter path to the least-cost results CSV file: ").strip().strip('"').strip("'")).expanduser()
    polygon_path = Path(input("Enter path to the cluster polygon vector file: ").strip().strip('"').strip("'")).expanduser()

    if not csv_path.exists():
        raise FileNotFoundError(f"CSV file not found: {csv_path}")
    if csv_path.suffix.lower() != ".csv":
        raise ValueError(f"Expected a .csv file, got: {csv_path.suffix}")

    if not polygon_path.exists():
        raise FileNotFoundError(f"Polygon file not found: {polygon_path}")

    allowed_vector_exts = {".shp", ".gpkg", ".geojson", ".json", ".parquet", ".fgb"}
    if polygon_path.suffix.lower() not in allowed_vector_exts:
        raise ValueError(
            f"Unsupported vector file type: {polygon_path.suffix}. "
            f"Allowed: {sorted(allowed_vector_exts)}"
        )

    return csv_path, polygon_path

In [5]:
least_cost_csv_path, clusters_path = get_user_input_paths()

iep_result = pd.read_csv(least_cost_csv_path)
clusters = gpd.read_file(clusters_path)

Enter path to the least-cost results CSV file:  "C:\Users\alexl\Dropbox\Self-employment\SEforALL\Work\Mozambique\Cooling_Module\GIS_data\IEP_LeastCost\Baseline_Results_April_2026.csv"
Enter path to the cluster polygon vector file:  "C:\Users\alexl\Dropbox\Self-employment\SEforALL\Work\Mozambique\Cooling_Module\GIS_data\Settlements\Mozambique_clusters.shp"


### Merge IEP Least-Cost result columns to the clusters

In [6]:
start_year = 2024
inter_year = None   # set to None if not applicable
end_year = 2030

# Delete any existing FinalElecCode* columns from clusters
existing_final_cols = [col for col in clusters.columns if col.startswith("FinalElecCode")]

if existing_final_cols:
    print(f"Deleting existing columns from clusters: {existing_final_cols}")
    clusters = clusters.drop(columns=existing_final_cols)
else:
    print("No existing FinalElecCode* columns found in clusters.")

# Build list of columns to merge from iep_result
year_columns = ["id", f"FinalElecCode{start_year}", f"FinalElecCode{end_year}"]

if inter_year is not None:
    year_columns.insert(2, f"FinalElecCode{inter_year}")

# Check required columns exist in iep_result
missing = [col for col in year_columns if col not in iep_result.columns]
if missing:
    raise ValueError(f"Missing columns in iep_result: {missing}")

# Merge onto cleaned clusters
iep_clusters = clusters.merge(iep_result[year_columns], on="id", how="left")

No existing FinalElecCode* columns found in clusters.


#### Translate Electrification code to strings

In [7]:
elec_code_map = {
    1:  'Grid Densification',
    10: 'Grid Extension',
    3:  'SHS',
    7:  'Mini Grid Hydro',
    8:  'Mini Grid PV',
    9:  'Mini Grid Wind',
    99: 'Non Electrified'
}

In [8]:
for yr in [start_year, inter_year, end_year]:
    if yr is not None:
        src_col = f'FinalElecCode{yr}'
        new_col = f'IEP_Result{yr}'
        if src_col in iep_clusters.columns:
            iep_clusters[new_col] = iep_clusters[src_col].map(elec_code_map)

#### Correct format for gep-dataframe

In [9]:
# Drop fid
iep_clusters = iep_clusters.drop(columns=['fid'], errors='ignore')

# Numeric formatting
iep_clusters['NL']         = iep_clusters['NL'].round(4).astype(float)
iep_clusters['Population'] = iep_clusters['Population'].round(2).astype(float)
iep_clusters['Area']       = iep_clusters['Area'].round(4).astype(float)
iep_clusters['Buildings']  = iep_clusters['Buildings'].round().astype('Int64')
iep_clusters['id']         = iep_clusters['id'].astype('Int64')

# FinalElecCode columns
elec_code_cols = [f'FinalElecCode{yr}' for yr in [start_year, inter_year, end_year] if yr is not None]
for col in elec_code_cols:
    if col in iep_clusters.columns:
        iep_clusters[col] = iep_clusters[col].round().astype('Int64')

# FinalElecTech columns
elec_code_cols = [f'IEP_Result{yr}' for yr in [start_year, inter_year, end_year] if yr is not None]
for col in elec_code_cols:
    if col in iep_clusters.columns:
        iep_clusters[col] = iep_clusters[col].astype(str)

# String columns
iep_clusters['Country'] = iep_clusters['Country'].astype(str).str.strip()
iep_clusters['DEGURBA'] = iep_clusters['DEGURBA'].astype(str).str.strip()

In [10]:
iep_clusters.head(4)

,NL,Population,Buildings,DEGURBA,Area,id,Country,geometry,FinalElecCode2024,FinalElecCode2030,IEP_Result2024,IEP_Result2030
0,0.0,88.13,144,Low Density Rural,0.521,4,Mozambique,"POLYGON ((31.47088 -22.47396, 31.47112 -22.473...",99,3,Non Electrified,SHS
1,0.0,6.34,14,Low Density Rural,0.032,5,Mozambique,"POLYGON ((31.47367 -22.47286, 31.47269 -22.473...",99,3,Non Electrified,SHS
2,0.0,7.37,6,Low Density Rural,0.031,9,Mozambique,"POLYGON ((31.42118 -22.45659, 31.42118 -22.456...",99,3,Non Electrified,SHS
3,0.0,117.46,339,Low Density Rural,0.650,10,Mozambique,"POLYGON ((31.41033 -22.45221, 31.41058 -22.452...",99,3,Non Electrified,SHS


## OPTIONAL!!

To keep things manageable in size, we set a threshold for the clusters to be processed related to the number of buildings.
Feel free to comment this out if not useful or needed in your analysis.

In [11]:
iep_clusters = iep_clusters.loc[iep_clusters["Buildings"] >= 100].copy()

## Export results for further processing on AgCap

In [12]:
## Export settlements to geopackage
iep_clusters.to_file(project_root / f"data/raw/settlements/input_file/Moz_settle_iep_{datetime.now().strftime('%Y%m%d')}.gpkg", driver="GPKG")
#iep_clusters.to_file(project_root / f"data/raw/settlements/input_file/Moz_settle_iep_{datetime.now().strftime('%Y%m%d')}", driver="GeoJSON")
#iep_clusters.to_parquet(project_root / f"data/raw/settlements/input_file/Moz_settle_iep_{datetime.now().strftime('%Y%m%d')}.parquet")